# 가중 평균 평점 구하기

In [36]:
import pandas as pd

In [3]:
df = pd.read_csv('review-vc_sales_by_collection.csv')

In [4]:
df['year'] = df['yr_month'].astype(str).str[:4].astype(int)

In [5]:
df

,yr_month,financial_category,collection,written_avg_rating,written_12_cnt,witten_all_cnt,written_12_ratio,all_avg_rating,all_12_cnt,all_all_cnt,all_12_ratio,sales_amount,sales_qty,year
0,202201,Box Springs,__TOTAL__,4.115207,77.0,434.0,0.177419,NaN,NaN,NaN,NaN,6916730.26,50316.0,2022
1,202201,Box Springs,Walter 9in,3.500000,3.0,8.0,0.375000,NaN,NaN,NaN,NaN,91988.82,446.0,2022
2,202201,Box Springs,Walter 7.5in,4.500000,1.0,10.0,0.100000,NaN,NaN,NaN,NaN,271300.75,1670.0,2022
3,202201,Box Springs,Walter 4in,4.666667,1.0,9.0,0.111111,NaN,NaN,NaN,NaN,229874.12,1231.0,2022
4,202201,Box Springs,Victor 9in,5.000000,0.0,2.0,0.000000,NaN,NaN,NaN,NaN,60917.50,297.0,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23388,202503,Toppers,1.5in GTFT w WonderBox,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,4739.44,117.0,2025
23389,202503,Toppers,1.5in GT MF,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,2635.14,73.0,2025
23390,202503,Toppers,1.5in Copper Convoluted,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,2025
23391,202503,Toppers,1.25in Swirl Copper,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,67.94,1.0,2025


In [6]:
#collection별, 연도별 평균 평점과 매출 합계
collection_yearly = df.groupby(['collection','year']).agg({
    'written_avg_rating': 'mean',
    'sales_amount': 'sum'
}).reset_index()

In [7]:
collection_yearly

,collection,year,written_avg_rating,sales_amount
0,0.75in Rejuvenator,2022,5.000000,8.179930e+03
1,0.75in Rejuvenator,2023,NaN,1.443330e+03
2,0.75in Rejuvenator,2024,NaN,0.000000e+00
3,0.75in Rejuvenator,2025,NaN,0.000000e+00
4,1.25in Swirl Copper,2022,2.666667,2.407901e+04
...,...,...,...,...
2503,Zaalonge Console Table,2025,NaN,0.000000e+00
2504,__TOTAL__,2022,3.908572,6.599888e+08
2505,__TOTAL__,2023,3.903510,5.320267e+08
2506,__TOTAL__,2024,3.827819,3.411127e+08


In [8]:
# 년도별 피벗 
collection_yearly_ratings = collection_yearly.pivot(index='collection', columns='year', values='written_avg_rating')
collection_yearly_sales = collection_yearly.pivot(index='collection', columns='year', values='sales_amount')

In [9]:
# 컬럼 이름 변경 
#collection_yearly_ratings.columns = [f'{year}_rating' if isinstance(col, int) or col.isdiigit() else col
#                                     for col in collection_yearly_ratings.columns]
#collection_yearly_sales.columns = [f'{year}_sales' 
#                                   for year in collection_yearly_sales.columns]

In [10]:
collection_yearly_ratings.columns = [
    f"{col}_rating" if isinstance(col, int) or col.isdigit() else col
    for col in collection_yearly_ratings.columns
]

# sales 컬럼 이름 정리 (연도 숫자일 경우에만 '_sales' 붙이기)
collection_yearly_sales.columns = [
    f"{col}_sales" if isinstance(col, int) or col.isdigit() else col
    for col in collection_yearly_sales.columns
]

In [11]:
collection_yearly_ratings.columns

Index(['2022_rating', '2023_rating', '2024_rating', '2025_rating'], dtype='object')

In [12]:
final_collection_yearly = pd.concat([collection_yearly_ratings, collection_yearly_sales], axis=1)

In [13]:
# 연도순 정렬
ordered_columns = sorted(final_collection_yearly, key=lambda x: (x.split('_')[0], x.split('_')[1]))

In [14]:
final_collection_yearly = final_collection_yearly[ordered_columns]

In [33]:
final_collection_yearly.fillna(0).to_csv('collection_yearly_out1.csv')

In [16]:
final_collection_yearly

,2022_rating,2022_sales,2023_rating,2023_sales,2024_rating,2024_sales,2025_rating,2025_sales
collection,,,,,,,,
0.75in Rejuvenator,5.000000,8.179930e+03,NaN,1.443330e+03,NaN,0.000000e+00,NaN,0.00
1.25in Swirl Copper,2.666667,2.407901e+04,3.000000,2.298572e+04,1.000000,1.220170e+03,NaN,136.05
1.5in Copper Convoluted,NaN,0.000000e+00,NaN,0.000000e+00,5.000000,0.000000e+00,NaN,0.00
1.5in GT MF,3.344252,4.206828e+05,3.176687,4.597973e+05,3.600000,1.195918e+05,2.000000,7070.83
1.5in GTFT w WonderBox,NaN,NaN,NaN,3.221500e+02,3.666667,4.152080e+04,2.000000,22397.73
...,...,...,...,...,...,...,...,...
Wood Squre Dining Table,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.00
Yelena 14in,3.842190,4.370094e+06,3.537253,2.765323e+06,3.810913,6.557629e+05,3.416667,25407.47
Yelena 18in,4.803333,2.956699e+05,4.196970,3.582455e+05,4.285714,1.166290e+05,NaN,0.00
